# WaterDRoP Automation

This notebook checks the active Python kernel/GPU environment, then batch-processes CSV files from an absolute input folder and writes one processed result file per input CSV.

## 1. Kernel and GPU Check

In [1]:
import chemprop
import lightning
import numpy as np
import pandas as pd
import torch
from chemprop import data, featurizers, models
from lightning import pytorch as pl
from pathlib import Path
from tqdm.notebook import tqdm

print("Python kernel is active.")
print(f"Chemprop: {chemprop.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"Lightning: {lightning.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    current_device = torch.cuda.current_device()
    print(f"CUDA device count: {torch.cuda.device_count()}")
    print(f"Current CUDA device: {current_device}")
    print(f"CUDA device name: {torch.cuda.get_device_name(current_device)}")
else:
    print("Running on CPU.")

Python kernel is active.
Chemprop: 2.2.3
PyTorch: 2.11.0+cu130
Lightning: 2.6.1
CUDA available: True
CUDA device count: 1
Current CUDA device: 0
CUDA device name: NVIDIA GeForce RTX 5070 Laptop GPU


## 2. Batch Prediction Automation

In [2]:
# Use absolute paths for the input and output folders.
PROJECT_ROOT = Path("/home/cenking/VsCode/WaterDRoP")
MODEL_DIR = PROJECT_ROOT / "model"

INPUT_DIR = Path("/home/cenking/VsCode/TEMP/xxx")
OUTPUT_DIR = Path("/home/cenking/VsCode/TEMP")

# Set to True to skip CSV files whose *_processed.csv output already exists.
SKIP_EXISTING = True

if not INPUT_DIR.is_absolute():
    raise ValueError("INPUT_DIR must be an absolute path.")
if not OUTPUT_DIR.is_absolute():
    raise ValueError("OUTPUT_DIR must be an absolute path.")
if not INPUT_DIR.exists():
    raise FileNotFoundError(f"INPUT_DIR does not exist: {INPUT_DIR}")
if not INPUT_DIR.is_dir():
    raise NotADirectoryError(f"INPUT_DIR is not a directory: {INPUT_DIR}")
if not MODEL_DIR.exists():
    raise FileNotFoundError(f"MODEL_DIR does not exist: {MODEL_DIR}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

non_csv_files = [path for path in INPUT_DIR.iterdir() if path.is_file() and path.suffix.lower() != ".csv"]
if non_csv_files:
    raise ValueError(
        "INPUT_DIR must contain only CSV files. Non-CSV files found:\n"
        + "\n".join(str(path) for path in non_csv_files)
    )

csv_files = sorted(path for path in INPUT_DIR.iterdir() if path.is_file() and path.suffix.lower() == ".csv")
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {INPUT_DIR}")

print(f"Input folder: {INPUT_DIR}")
print(f"Output folder: {OUTPUT_DIR}")
print(f"CSV files found: {len(csv_files)}")

Input folder: /home/cenking/VsCode/TEMP/xxx
Output folder: /home/cenking/VsCode/TEMP
CSV files found: 1


In [3]:
filenames_stg1 = [MODEL_DIR / f"wd_stg1_n{i}.pt" for i in range(1, 5)]
filenames_stg2 = [MODEL_DIR / f"wd_stg2_n{i}.pt" for i in range(1, 5)]
required_model_files = filenames_stg1 + filenames_stg2

missing_model_files = [path for path in required_model_files if not path.exists()]
if missing_model_files:
    raise FileNotFoundError(
        "Missing model files:\n" + "\n".join(str(path) for path in missing_model_files)
    )

stg1_models = [models.MPNN.load_from_file(str(path)) for path in filenames_stg1]
stg2_models = [models.MPNN.load_from_file(str(path)) for path in filenames_stg2]

accelerator = "gpu" if torch.cuda.is_available() else "cpu"
trainer_kwargs = {
    "accelerator": accelerator,
    "devices": 1,
    "logger": False,
    "enable_checkpointing": False,
    "enable_progress_bar": False,
}

print(f"Loaded {len(stg1_models)} Stage 1 models and {len(stg2_models)} Stage 2 models.")
print(f"Prediction accelerator: {accelerator}")

Loaded 4 Stage 1 models and 4 Stage 2 models.
Prediction accelerator: gpu


In [4]:
def _format_half_life(log_rate):
    return f'{float(f"{(np.log(2) / np.exp(log_rate)):.2g}"):g}'


def _validate_input_frame(df_input, input_file):
    if list(df_input.columns) != ["SMILES"]:
        raise ValueError(f'{input_file} must contain exactly one column named "SMILES".')
    if df_input.empty:
        raise ValueError(f"{input_file} contains no SMILES rows.")
    if df_input["SMILES"].isna().any():
        raise ValueError(f'{input_file} contains blank values in the "SMILES" column.')


def predict_smiles_batch(smis):
    featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()
    ys = np.random.rand(len(smis))
    all_data = [data.MoleculeDatapoint.from_smi(smi, y) for smi, y in zip(smis, ys)]
    test_dset = data.MoleculeDataset(all_data, featurizer)
    test_loader = data.build_dataloader(test_dset, shuffle=False, batch_size=len(smis))

    predictions = []
    for model in stg1_models:
        with torch.inference_mode():
            trainer = pl.Trainer(**trainer_kwargs)
            preds = torch.concat(trainer.predict(model, test_loader)).detach().cpu().numpy()
            predictions.append(preds)

    predictions = np.array(predictions)
    stg1_preds = predictions.mean(axis=0)
    ranges_lo_stg1 = predictions.max(axis=0)
    ranges_hi_stg1 = predictions.min(axis=0)

    predictions = []
    for model in stg2_models:
        with torch.inference_mode():
            trainer = pl.Trainer(**trainer_kwargs)
            preds = torch.concat(trainer.predict(model, test_loader)).detach().cpu().numpy()
            predictions.append(preds)

    predictions = np.array(predictions)
    stg2_preds = predictions.mean(axis=0)
    ranges_lo_stg2 = predictions.max(axis=0)
    ranges_hi_stg2 = predictions.min(axis=0)

    final_preds = []
    ranges_lo_str = []
    ranges_hi_str = []
    threshold = np.log(np.log(2) / 365)

    for i in range(len(smis)):
        if stg1_preds[i][0] < threshold or stg2_preds[i][0] < threshold:
            final_preds.append("> 1 year")
            ranges_lo = ranges_lo_stg1[i][0]
            ranges_hi = ranges_hi_stg1[i][0]
        else:
            final_preds.append(_format_half_life(stg2_preds[i][0]))
            ranges_lo = ranges_lo_stg2[i][0]
            ranges_hi = ranges_hi_stg2[i][0]

        ranges_lo_str.append("> 1 year" if ranges_lo < threshold else _format_half_life(ranges_lo))
        ranges_hi_str.append("> 1 year" if ranges_hi < threshold else _format_half_life(ranges_hi))

    return pd.DataFrame(
        {
            "SMILES": smis,
            "Half-life (days)": final_preds,
            "Range, low": ranges_lo_str,
            "Range, high": ranges_hi_str,
        }
    )


def predict_file(input_file, output_dir):
    output_file = output_dir / f"{input_file.stem}_processed.csv"
    if SKIP_EXISTING and output_file.exists():
        return output_file

    df_input = pd.read_csv(input_file)
    _validate_input_frame(df_input, input_file)

    smis = df_input["SMILES"].astype(str).values
    output = predict_smiles_batch(smis)
    output.to_csv(output_file, index=False)
    return output_file

In [5]:
processed_files = []

for input_file in tqdm(csv_files, desc="Processing files", unit="file"):
    processed_files.append(predict_file(input_file, OUTPUT_DIR))

print(f"Processed files: {len(processed_files)}")
print(f"Results written to: {OUTPUT_DIR}")

Processing files:   0%|          | 0/1 [00:00<?, ?file/s]

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA GeForce RTX 5070 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/cenking/miniconda3/envs/WaterDRoP/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/cenking/miniconda3/envs/WaterDRoP/lib/

Processed files: 1
Results written to: /home/cenking/VsCode/TEMP
